In [1]:
#import kagglehub
# https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset?resource=download
# Download latest version
#path = kagglehub.dataset_download("suraj520/customer-support-ticket-dataset")
#print("Path to dataset files:", path)

In [2]:
from pathlib import Path
import pandas as pd
import os
from openai import OpenAI
import dotenv
dotenv.load_dotenv()

# Find repo root
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("pyproject.toml not found")

REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "assets" / "datasets"

# Load the dataset
file_path = DATA_DIR / "customer_support_tickets.csv"
df = pd.read_csv(file_path)

### Scaffolding

In [ ]:
from pathlib import Path
import json
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

import pandas as pd
import torch
from src.support import get_device
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)

from peft import PromptTuningConfig, get_peft_model, PeftModel, TaskType


# -------------------------
# Repo root + data path
# -------------------------
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("pyproject.toml not found")

REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "assets" / "datasets"
CSV_PATH = DATA_DIR / "customer_support_tickets.csv"


# -------------------------
# Labels - from dataset
# -------------------------
TYPE_LABELS = [
    "Billing inquiry",
    "Cancellation request",
    "Product inquiry",
    "Refund request",
    "Technical issue",
]
PRIORITY_LABELS = ["Critical", "High", "Medium", "Low"]


# -------------------------
# Prompt rules
# -------------------------


SYSTEM_RULES = f"""
You are a ticket router. Return ONLY a JSON object.
No markdown. No backticks. No extra text.
Keys must be exactly: type, routing_summary.

type must be exactly one of:
{'\n'.join(f'- {t}' for t in TYPE_LABELS)}
""".strip()


# -------------------------
# Model + device (Mac MPS)
# -------------------------
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

DEVICE = get_device()


# -------------------------
# Data prep
# -------------------------
def build_input_text(row: pd.Series) -> str:
    subject = str(row.get("Ticket Subject", "")).strip()
    product = str(row.get("Product Purchased", "")).strip()
    desc = str(row.get("Ticket Description", "")).strip()
    return f"Subject: {subject}\nProduct: {product}\nDescription: {desc}".strip()

def build_target_json(row: pd.Series) -> str:
    obj = {
        "type": str(row["Ticket Type"]).strip(),
#        "priority": str(row["Ticket Priority"]).strip(),
#        "routing_summary": f"Route as {row['Ticket Type']} with {row['Ticket Priority']} priority."
    }
    return json.dumps(obj, ensure_ascii=False)

def load_and_split(csv_path: Path, test_size=0.2, seed=42):
    df = pd.read_csv(csv_path)

    # Mask out rows with NULL or invalid values
    required = ["Ticket Type", "Ticket Priority", "Ticket Subject", "Product Purchased", "Ticket Description"]
    mask = (
        df[required].notna().all(axis=1)
        & df["Ticket Type"].isin(TYPE_LABELS)
        #& df["Ticket Priority"].isin(PRIORITY_LABELS)
    )
    df = df.loc[mask].copy()

    # Build the input text and target JSON fields
    df["input_text"] = df.apply(build_input_text, axis=1)
    df["target_json"] = df.apply(build_target_json, axis=1)

    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=df["Ticket Type"]
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


# -------------------------
# Prompts: zero-shot / few-shot
# -------------------------
def make_prompt(
    user_text: str,
    examples: Optional[List[Tuple[str, str]]] = None,
) -> str:
    base = f"""{SYSTEM_RULES}
Return ONLY the JSON object. No commentary. No code block.
"""
    if examples:
        shots = "\n\n".join(
            [f"Input ticket:\n{ex_in}\nJSON:\n{ex_out}" for ex_in, ex_out in examples]
        )
        base += f"""Here are examples: \n{shots}. \nNow classify this ticket."""

    return base + f"""Input ticket: {user_text}\nJSON:"""


# -------------------------
# Generation + parsing
# -------------------------
JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

def extract_json(text: str) -> Dict:
    matches = list(JSON_RE.finditer(text))
    if not matches:
        raise ValueError("No JSON object found")
    return json.loads(matches[-1].group(0))


@torch.inference_mode()
def generate_text(model, tokenizer, prompt: str, max_new_tokens=160) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = out[0, prompt_len:]              # only new tokens
    return tokenizer.decode(gen_ids, skip_special_tokens=True)

def score_predictions(golds: List[Dict], preds: List[Optional[Dict]]) -> Dict[str, float]:
    valid = []
    type_enum_ok = []
    #prio_enum_ok = []

    y_type, p_type = [], []
    y_prio, p_prio = [], []

    for g, p in zip(golds, preds):
        is_valid = p is not None
        valid.append(1 if is_valid else 0)

        if not is_valid:
            type_enum_ok.append(0)
            #prio_enum_ok.append(0)
            continue

        t = p.get("type")
        #r = p.get("priority")

        type_enum_ok.append(1 if t in TYPE_LABELS else 0)
        #prio_enum_ok.append(1 if r in PRIORITY_LABELS else 0)

        if t in TYPE_LABELS:
            y_type.append(g["type"])
            p_type.append(t)
        #if r in PRIORITY_LABELS:
        #    y_prio.append(g["priority"])
        #    p_prio.append(r)

    out = {
        "json_valid_rate": sum(valid) / len(valid),
        "type_enum_valid_rate": sum(type_enum_ok) / len(type_enum_ok),
        #"priority_enum_valid_rate": sum(prio_enum_ok) / len(prio_enum_ok),
    }
    out["type_accuracy"] = accuracy_score(y_type, p_type) if y_type else 0.0
    out["type_macro_f1"] = f1_score(y_type, p_type, average="macro") if y_type else 0.0
    #out["priority_accuracy"] = accuracy_score(y_prio, p_prio) if y_prio else 0.0
    #out["priority_macro_f1"] = f1_score(y_prio, p_prio, average="macro") if y_prio else 0.0
    return out


# -------------------------
# Dataset for prompt tuning
# -------------------------
@dataclass
class LMDataset(torch.utils.data.Dataset):
    prompts: List[str]
    targets: List[str]
    tokenizer: AutoTokenizer
    max_length: int = 256

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        target = self.targets[idx]

        full = prompt + target
        tok_full = self.tokenizer(
            full,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )

        tok_prompt = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )
        prompt_len = len(tok_prompt["input_ids"])

        labels = tok_full["input_ids"].copy()
        labels[:min(prompt_len, len(labels))] = [-100] * min(prompt_len, len(labels))
        tok_full["labels"] = labels

        # IMPORTANT: return lists, not torch tensors
        return tok_full


# -------------------------
# Train prompt tuning
# -------------------------

@dataclass
class CausalLMDataCollatorPadLabels:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # 1) Separate labels so tokenizer.pad doesn't try to tensorize ragged lists
        labels = [f.pop("labels") for f in features]

        # 2) Pad model inputs
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt",
        )

        # 3) Pad labels to match input length
        max_len = batch["input_ids"].shape[1]
        padded_labels = []
        for lab in labels:
            lab = lab[:max_len]
            lab = lab + [-100] * (max_len - len(lab))
            padded_labels.append(lab)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)

        return batch


def train_prompt_tuning(train_df: pd.DataFrame, output_dir: str = "./pt_supportrouter"):
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, local_files_only=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.float16 if DEVICE in ["cuda", "mps"] else torch.float32,
        local_files_only=True,
    ).to(DEVICE)

    pt_config = PromptTuningConfig(
        task_type=TaskType.CAUSAL_LM,
        num_virtual_tokens=20,
        tokenizer_name_or_path=BASE_MODEL,
    )
    model = get_peft_model(base_model, pt_config)
    model.print_trainable_parameters()

    prompts = [make_prompt(x) for x in train_df["input_text"].tolist()]
    targets = [t for t in train_df["target_json"].tolist()]
    ds_train = LMDataset(prompts=prompts, targets=targets, tokenizer=tokenizer, max_length=384)

    # Define collator BEFORE Trainer
    collator = CausalLMDataCollatorPadLabels(tokenizer)

    args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=5e-3,
        num_train_epochs=1,
        logging_steps=50,
        save_strategy="epoch",
        report_to="none",
        fp16=False,
        bf16=False,
        dataloader_pin_memory=False,  # MPS warning fix
    )

    trainer = Trainer(model=model, args=args, train_dataset=ds_train, data_collator=collator)
    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    return output_dir


def load_prompt_tuned_model(pt_dir: str):
    tokenizer = AutoTokenizer.from_pretrained(pt_dir, use_fast=True, local_files_only=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.float16 if DEVICE in ["cuda", "mps"] else torch.float32,
        local_files_only=True,
    ).to(DEVICE)
    model = PeftModel.from_pretrained(base_model, pt_dir).to(DEVICE)
    return model, tokenizer


# -------------------------
# Eval runners
# -------------------------
def eval_model(
    test_df,
    model,
    tokenizer,
    n: int = 200,
    examples: Optional[List[Tuple[str, str]]] = None,
):
    sub = test_df.iloc[:n]
    golds = [json.loads(t) for t in sub["target_json"].tolist()]

    preds = []
    for x in sub["input_text"].tolist():
        prompt = make_prompt(x, examples=examples)
        gen = generate_text(model, tokenizer, prompt)
        try:
            preds.append(extract_json(gen))
        except Exception:
            preds.append(None)

    return score_predictions(golds, preds)


In [4]:
train_df, test_df = load_and_split(CSV_PATH)
# Base model for baselines
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, local_files_only=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16 if DEVICE in ["cuda", "mps"] else torch.float32,
    local_files_only=True,
).to(DEVICE)



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:

print("DEVICE:", DEVICE)


print("Zero-shot:", eval_model(test_df, base, tok, n=200))

ex_df = train_df.sample(n=4, random_state=42)
examples = list(zip(ex_df["input_text"].tolist(), ex_df["target_json"].tolist()))
print("Few-shot:", eval_model(test_df, base, tok, n=200, examples=examples))


# Train prompt tuning 
pt_dir = train_prompt_tuning(train_df, output_dir="./pt_supportrouter")
pt_model, pt_tok = load_prompt_tuned_model(pt_dir)
print("Prompt-tuned:", eval_model(test_df, pt_model, pt_tok, n=200))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


DEVICE: mps
Zero-shot: {'json_valid_rate': 0.995, 'type_enum_valid_rate': 0.995, 'type_accuracy': 0.20603015075376885, 'type_macro_f1': 0.1593760769701343}
Few-shot: {'json_valid_rate': 0.995, 'type_enum_valid_rate': 0.99, 'type_accuracy': 0.2222222222222222, 'type_macro_f1': 0.18130733969684315}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


trainable params: 40,960 || all params: 3,085,979,648 || trainable%: 0.0013


Step,Training Loss
50,2.081300
100,0.293300
150,0.256900
200,0.242400
250,0.237200
300,0.236000
350,0.233900
400,0.234000


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/opt/anaconda3/envs/ai_backup_py312/lib/python3.12/site-packages/peft/peft_model.py:2066: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


Prompt-tuned: {'json_valid_rate': 0.265, 'type_enum_valid_rate': 0.265, 'type_accuracy': 0.2641509433962264, 'type_macro_f1': 0.23632880529432257}


In [6]:
for i in range(5):
    x = test_df.iloc[i]["input_text"]
    gen = generate_text(pt_model, pt_tok, make_prompt(x))
    print("----\n", gen)

/opt/anaconda3/envs/ai_backup_py312/lib/python3.12/site-packages/peft/peft_model.py:2066: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


----
 {"type": "Technical issue"} 1
```json
{"type": "Technical issue"}
```
----
 {"type": "Refund request"}  No markdown. No backticks. No extra text.
Keys must be exactly: type, routing_summary.

type must be exactly one of:
- Billing inquiry
- Cancellation request
- Product inquiry
- Refund request
- Technical issue
No JSON. No extra text.
{"type": "Refund request"}  No markdown. No backticks. No extra text.
----
 {"type": "Cancellation request"} 1
```json
{"type": "Billing inquiry"}
```
----
 {"type": "Billing inquiry"}  {"type": "Billing inquiry"}
----
 {"type": "Cancellation request"}  {"type": "Cancellation request"}
